### Import Dependencies

In [ ]:
import openai
from qdrant_client import QdrantClient

### Define embedding function

In [ ]:
def get_embedding(text):
    response = openai.embeddings.create( # was client before
        input=text,
        model="text-embedding-3-small"
    )
    return response.data[0].embedding

### Retrieval Function

In [ ]:
qdrant_client = QdrantClient(
    url="http://localhost:6333"
)

In [ ]:
def retrieve_data(query, k=5): # 5 most similar items to users query
    embedding = get_embedding(query) # so we are actually creating related vector here
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=embedding,
        limit=k
    )
    
    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []
    
    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload.get("preprocessed_description"))
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload.get("average_rating"))
    
    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [ ]:
retrieved_context = retrieve_data("Do you have a USB connectable fan for hot summers.", k=10)
retrieved_context

### Format retrieved context

In [ ]:
def process_context(context):
    formatted_context = ""
    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"
    return formatted_context
 

In [ ]:
preprocessed_context = process_context(retrieved_context)
print(preprocessed_context)

### Create prompt template function

In [ ]:
def build_prompt(preprocessed_context, question): # question is the users question...
    prompt = f"""You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}
"""
    return prompt


In [ ]:
# we will need to take the user question and 

#dataForQuestion = retrieve_data(question) \n
#questionProcessedContext = preprocess_context(dataForQuestion)\n
#prompt = buildPrompt(questionProcessedContext, question)\n

In [ ]:
prompt = build_prompt(preprocessed_context, "Do you have a USB connectable fan for hot summers.")

print(prompt)

### Generate answer

In [ ]:
def generate_answer(prompt):
    response = openai.chat.completions.create(
        model="gpt-5-nano",
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning_effort="minimal"
    )
    return response.choices[0].message.content

In [ ]:
print(generate_answer(prompt))

In [ ]:
question = "Do you have laptops"
data_for_question = retrieve_data(question)

processed_context = process_context(data_for_question)
prompt = build_prompt(processed_context, question)
print(generate_answer(prompt))

In [ ]:
def rag_pipeline(question, topk_k=5):
    data_for_question = retrieve_data(question, k=topk_k)
    processed_context = process_context(data_for_question)
    prompt = build_prompt(processed_context, question)
    return generate_answer(prompt)

print(rag_pipeline("Do you have a USB connectable fan for hot summers."))

In [ ]:
print(rag_pipeline("I need headphones that have a microphone and a rating above 4"))

In [ ]:
print(rag_pipeline("I need headphones that have a microphone and a rating below 4"))